# Pulse scoring on a GPU

Scores the Pulse recordings waiting on the server with the same engine as the nightly run, on
Kaggle's free GPU. Whisper (the English part) runs on the GPU; IndicConformer (Hindi reading)
runs on the CPU, as in the nightly run. Before running:

1. **Settings → Accelerator:** GPU T4.
2. **Settings → Internet:** on.
3. **Add-ons → Secrets:** add `ENGINE_REPOSITORY`, `SUPABASE_URL`, `SUPABASE_SECRET_KEY` and
   `HF_TOKEN` (see the setup guide, Part 6), and tick them for this notebook.

The engine is installed, with the exact library versions of the nightly run, before the project's
secrets are read; they are then given to the scoring run alone. Nothing secret is written in this
notebook or in its output.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name: str) -> str:
    try:
        return secrets.get_secret(name)
    except Exception:
        raise RuntimeError(f"Add the secret {name} and tick it for this notebook.") from None


# Outside /kaggle/working, which Kaggle keeps with the notebook's versions.
SOURCE = Path("/tmp/agla-kadam")
shutil.rmtree(SOURCE, ignore_errors=True)
# The address may hold a read token: it is used for cloning only, never printed, and removed from
# the copy once cloned.
clone = subprocess.run(
    ["git", "clone", "--quiet", "--depth", "1", secret("ENGINE_REPOSITORY"), str(SOURCE)],
    capture_output=True,
)
if clone.returncode != 0:
    raise RuntimeError("The repository could not be cloned; check the ENGINE_REPOSITORY secret.")
subprocess.run(["git", "-C", str(SOURCE), "remote", "remove", "origin"], check=True)

# PyTorch is already installed with GPU support on Kaggle; the other libraries are pinned.
engine = SOURCE / "engine"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--constraint",
        str(engine / "constraints.txt"),
        f"{engine}[speech]",
    ],
    check=True,
)

In [ ]:
environment = dict(os.environ)
for name in ("SUPABASE_URL", "SUPABASE_SECRET_KEY"):
    environment[name] = secret(name)
try:
    environment["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    print("HF_TOKEN is not set: Hindi reading will be transcribed with Whisper.")

# Scores until nothing is left or the time limit is reached; a later run continues.
subprocess.run(
    [
        sys.executable,
        "-m",
        "agla_engine",
        "score",
        "--runner",
        "gpu",
        "--device",
        "cuda",
        "--max-minutes",
        "600",
    ],
    env=environment,
    check=True,
)